# Davinci Notebook 11 — Evaluation Harness & Baselines

A practical framework for perplexity/loss evaluation, Persian text normalization diagnostics, task scoring, tool-call JSON validation, and reproducible test manifests. This is an evaluation scaffold; benchmark datasets require separate acquisition, licensing, and preprocessing.

In [ ]:
import torch, json, math, re, time, hashlib, unicodedata
from pathlib import Path


In [ ]:
import subprocess, sys\nREPO_ROOT = Path.cwd()\nif not (REPO_ROOT / 'src' / 'davinci').is_dir():\n    REPO_ROOT = Path('/content/Davinci')\n    if not (REPO_ROOT / 'src' / 'davinci').is_dir():\n        subprocess.run(['git', 'clone', 'https://github.com/keyhmoham/Davinci.git', str(REPO_ROOT)], check=True)\nsys.path.insert(0, str(REPO_ROOT))\nprint('Using Davinci source:', REPO_ROOT)\n

## 1. Text normalization and diagnostics

In [ ]:
def normalize_fa(s):\n    s = unicodedata.normalize('NFC', str(s)).replace('ي', 'ی').replace('ى', 'ی').replace('ك', 'ک').replace('ـ', '')\n    s = re.sub('[\\u200e\\u200f\\u202a-\\u202e\\u2066-\\u2069]', '', s)\n    return s\n\ndef script_stats(text):\n    return {'chars': len(text), 'persian_arabic_block': sum('\\u0600' <= c <= '\\u06ff' for c in text),\n            'latin': sum('a' <= c.lower() <= 'z' for c in text), 'digits': sum(c.isdigit() for c in text)}\n

from src.davinci.evaluation import (\n    evaluate_language_model, exact_match_metrics, grouped_exact_match_metrics,\n)\n\ndef evaluate_lm(model, loader, device, max_batches=None):\n    # Canonical implementation handles ignored labels, token-weighting, and mode restoration.\n    return evaluate_language_model(model, loader, device=device, max_batches=max_batches)\n\ndef evaluate_answers(predictions, references, groups):\n    # Group labels must come from the benchmark manifest; do not infer language automatically.\n    return grouped_exact_match_metrics(predictions, references, groups)\n\nprint('Canonical evaluation helpers loaded.')\n

In [ ]:
from src.davinci.evaluation import (\n    evaluate_language_model, exact_match_metrics, grouped_exact_match_metrics,\n)\n\ndef evaluate_lm(model, loader, device, max_batches=None):\n    # Canonical implementation handles ignored labels, token-weighting, and mode restoration.\n    return evaluate_language_model(model, loader, device=device, max_batches=max_batches)\n\ndef evaluate_answers(predictions, references, groups):\n    # Group labels must come from the benchmark manifest; do not infer language automatically.\n    return grouped_exact_match_metrics(predictions, references, groups)\n\nprint('Canonical evaluation helpers loaded.')\n

## 3. Tool-call format validation
Use a canonical JSON representation for generated calls; tool execution itself must be sandboxed and permission-controlled.

In [ ]:
def validate_tool_call(obj,available_tools):
 errors=[]
 if not isinstance(obj,dict): return ['Expected object']
 if obj.get('type')!='tool_call': errors.append('type must be tool_call')
 tool=obj.get('name'); args=obj.get('arguments')
 if tool not in available_tools: errors.append('unknown tool name')
 if not isinstance(args,dict): errors.append('arguments must be a JSON object')
 return errors
example={'type':'tool_call','name':'search','arguments':{'query':'example'}}
print('Example validation:',validate_tool_call(example,['search','browser']))


## 4. Benchmark run manifest
Record model revision, tokenizer revision, dataset identifiers, split, prompt format, sampling settings, and evaluator version for reproducibility.

In [ ]:
def write_eval_manifest(path, manifest):\n    manifest = dict(manifest)\n    manifest['created_utc'] = time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())\n    manifest['manifest_sha256'] = hashlib.sha256(json.dumps(manifest, sort_keys=True, ensure_ascii=False).encode()).hexdigest()\n    Path(path).write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding='utf-8')\n    return manifest\n\nsample_manifest = {'model': 'Davinci experimental', 'model_revision': 'fill-in', 'tokenizer': 'fill-in',\n                   'dataset': 'fill-in', 'split': 'test', 'seed': 42, 'prompt_template_version': 'v1'}\nprint('Fill the manifest fields with actual values; do not treat this placeholder as a benchmark result.')\n

## 5. Benchmarking checklist\n- Persian, English, and code-mixed results must be reported separately using explicit manifest labels.\n- Include sample counts and uncertainty intervals; do not compare tiny groups as if equally reliable.\n- Long-context retrieval: vary evidence location, distractors, and context length; report each length separately.\n- Tool selection, argument correctness, execution outcomes, and recovery.\n- Safety and refusal quality.\n- Contamination checks and licensing/provenance audit.\n- Report confidence intervals and multiple seeds where practical.\n